# 4.4 — Troubleshoot and Optimize Document Processing

**Exam domain:** Domain 4.0 — Document Processing · **Weight:** 15%

## The problem this solves

The pipeline has been running for a month. Nobody has looked at it, because nothing has errored.
Then finance notices that 14% of invoices have a `NULL` total, the monthly Snowflake bill has a line
item nobody can explain, and the one document type that matters most — the new vendor's redesigned
invoice — extracts almost nothing correctly.

None of those are exceptions. They are a pipeline behaving exactly as written. This notebook is
about making failure visible, making cost deliberate, and knowing what to do when the model itself
is the thing that is wrong.

## What you will be able to do

- Turn silent `NULL`s into rows that carry an error message you can query
- Read a "file not found" and know which of five causes you are looking at
- Name the levers that actually move the cost of a document pipeline, and the ones that don't
- Audit the privileges a document pipeline needs, in the order failures surface
- Decide when prompt work has run out and fine-tuning `arctic-extract` is the right answer

## Before you start

- Notebooks 4.1 to 4.3 — this one assumes the functions, the stage and the pipeline
- ACCOUNTADMIN for the `ACCOUNT_USAGE` queries and the grants
- A second role (the setup script creates `GENAI_ANALYST`) to test access with

📖 **Snowflake documentation for this notebook**
- [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)
- [Parsing documents with AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)
- [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)
- [Fine-tuning arctic-extract models](https://docs.snowflake.com/en/user-guide/snowflake-cortex/arctic-extract-finetuning)
- [AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens)
- [GET_PRESIGNED_URL](https://docs.snowflake.com/en/sql-reference/functions/get_presigned_url)
- [AI function privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


---
## Failures, and what each one actually looks like

The hard part of debugging document processing is that most failures are quiet. A missing file, a
malformed accessor and a genuinely unreadable scan all produce the same thing: a `NULL` in a column,
in a row that inserted successfully.

| Symptom | Cause | Fix |
|---|---|---|
| File not found in stage | Directory table stale after an upload outside Snowflake | `ALTER STAGE <name> REFRESH;` or `DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE)` |
| File access error on a path that exists | FILE objects are unsupported on user stages, table stages, internal stages with `TYPE = 'SNOWFLAKE_FULL'`, external stages with `AWS_CSE`/`AZURE_CSE`, and double-quoted stage names | Move the file to a supported named stage |
| Access denied on stage | Missing stage privilege | `GRANT READ ON STAGE …` (internal) / `GRANT USAGE ON STAGE …` (external) |
| Size limit exceeded | File larger than 100 MB | Split the file — `page_filter` does not help, the limit is on the file |
| Page limit exceeded | Over 2,000 pages for `AI_PARSE_DOCUMENT`, or over 125 for `AI_EXTRACT` | Split, or narrow with `page_filter` |
| Unsupported format | XLSX, CSV, video | Convert to PDF or a supported image format. DOCX and PPTX **are** supported |
| Bare `NULL` returned | Empty or corrupt file, or an internal failure | Pass `TRUE` as the third argument to see the error |
| `AI_EXTRACT` field is `NULL` | Reading `:field` instead of `:response:field`, or an ambiguous question | Fix the accessor; make the question specific and single-valued |
| Small text misread | Low-resolution scan | `config => {'scale_factor': 2.0}` on `AI_EXTRACT` — range 1.0 to 4.0 |
| Resolution limit | Image over 10,000 × 10,000 pixels | Downsample before staging |

There is no `TRY_AI_PARSE_DOCUMENT` and no `TRY_AI_EXTRACT`. The documented pattern is the trailing
`return_error_details` BOOLEAN on `AI_PARSE_DOCUMENT`, and the `error` key that `AI_EXTRACT` always
returns.

→ [More on parsing limits and errors](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)

## What actually moves the bill

| Lever | Effect |
|---|---|
| `page_filter` instead of whole documents | Parsing is billed per page. This is the largest lever there is |
| Cache parse results | Parse once, store the text, never re-parse a file whose `LAST_MODIFIED` has not changed |
| `AI_EXTRACT` on the FILE directly | One AI call instead of two, when you do not need the raw text |
| Pre-validate size and format | Keep files that cannot succeed out of the batch |
| `return_error_details => TRUE` | Not a cost lever — a visibility one. It is what tells you which pages you paid for and got nothing from |
| Warehouse no larger than MEDIUM | Snowflake's documented recommendation; larger warehouses do not increase performance for these calls |
| Batch, don't loop | One set-based statement over a directory table, not row-by-row procedure calls |

Two things that are *not* levers, and are commonly believed to be: choosing OCR over LAYOUT (both
bill per page at the same rate), and `extract_images` (documented as carrying no additional cost).


In [ ]:
%%sql -r error_handling_1
-- ============================================================
-- Error handling: return_error_details => TRUE gives
--   {"value": {...}, "error": ..., "metadata": {...}}
-- instead of a bare NULL, so a failed file becomes an auditable row.
-- ============================================================
SELECT
    RELATIVE_PATH,
    parsed:value:content::VARCHAR AS extracted_text,
    parsed:error                  AS parse_error,
    parsed:metadata               AS parse_metadata,
    IFF(parsed:error IS NULL AND parsed:value:content IS NOT NULL, 'OK', 'PARSE_FAILED') AS status
FROM (
    SELECT
        RELATIVE_PATH,
        AI_PARSE_DOCUMENT(
            TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', RELATIVE_PATH),
            {'mode': 'OCR'},
            TRUE                       -- return_error_details
        ) AS parsed
    FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE)
    WHERE LOWER(RELATIVE_PATH) LIKE '%.pdf'
      AND SIZE < 104857600
);


In [ ]:
%%sql -r error_handling_2
-- AI_EXTRACT needs no flag: its return always carries an `error` key alongside `response`.
-- Select both, so a failure is a value in a column rather than an absence.
SELECT
    RELATIVE_PATH,
    fields:response AS extracted,
    fields:error    AS extract_error
FROM (
    SELECT RELATIVE_PATH,
           AI_EXTRACT(
               file           => TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', RELATIVE_PATH),
               responseFormat => {'total': 'What is the total amount due?'}
           ) AS fields
    FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE)
    WHERE LOWER(RELATIVE_PATH) LIKE '%.pdf'
);


> ### ⚠️ Common misconceptions
>
> **"If parsing fails, the query fails, so I'll see it."**
> By default a failure returns `NULL`. The statement succeeds, the row inserts, and the only trace
> is an empty column that looks identical to a blank page. `return_error_details => TRUE` is the
> difference between a pipeline you can audit and one you have to trust.
> → [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)
>
> **"There must be a `TRY_` version, like `TRY_TO_DATE`."**
> There is no `TRY_AI_PARSE_DOCUMENT` or `TRY_AI_EXTRACT`. Writing one produces an unknown-function
> error, which at least fails loudly — unlike most of the mistakes in this notebook.
> → [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)
>
> **"A pre-signed URL is how I check whether a file is on the stage."**
> `GET_PRESIGNED_URL` returns a URL even when the file does not exist, so a working call proves
> nothing about the file. It also requires server-side encryption on the stage. Query the directory
> table to test existence.
> → [GET_PRESIGNED_URL](https://docs.snowflake.com/en/sql-reference/functions/get_presigned_url)
>
> **"Switching from LAYOUT to OCR will cut the parsing bill."**
> Both modes bill per page at the same rate. What changes is the size of the text that comes out,
> which affects the input tokens of whatever function reads it next. Optimising the mode to save
> parsing money is optimising the wrong number.
> → [Parsing documents: cost considerations](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)


### The three file references, when you are debugging

`TO_FILE` is what the AI functions consume. The two URL functions exist for humans, and they differ
in who counts as a human.

- `BUILD_SCOPED_FILE_URL` produces an encoded URL valid for the caller until the persisted query
  result period ends — currently 24 hours. It stays inside Snowflake authentication.
- `GET_PRESIGNED_URL` produces a storage-level URL that works for anyone holding it, with no
  Snowflake login. Default expiry 3,600 seconds; maximum 3,600 for AWS S3 via an IAM role and for
  Microsoft Fabric OneLake, 604,800 seconds elsewhere. It needs server-side encryption on the stage.

When a parse fails and you want to look at the document yourself, the pre-signed URL is the fast
answer and the one that leaves a copy of a customer contract in your browser history. Prefer the
scoped URL when the person who needs to look is a Snowflake user.

→ [More on scoped URLs](https://docs.snowflake.com/en/sql-reference/functions/build_scoped_file_url)


In [ ]:
%%sql -r url_debugging_1
-- GET_PRESIGNED_URL: usable OUTSIDE Snowflake, no Snowflake authentication required.
--   GET_PRESIGNED_URL( @<stage>, '<relative_path>' [, <expiration_seconds> ] )
--   Default 3600 s. Maximum depends on the storage:
--     AWS S3 via an IAM role / Microsoft Fabric OneLake -> 3,600 s
--     other external stages                             -> 604,800 s (7 days)
--   Requires server-side encryption on the stage.
--   Privilege: USAGE (external stage) or READ (internal stage).
SELECT
    RELATIVE_PATH,
    GET_PRESIGNED_URL('@GENAI_STUDY.PUBLIC.DOCS_STAGE', RELATIVE_PATH, 3600) AS debug_url,
    ROUND(SIZE / 1048576.0, 2) AS size_mb,
    IFF(SIZE > 104857600, 'OVERSIZED - skip', 'OK') AS size_check
FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE);


In [ ]:
%%sql -r url_debugging_2
-- BUILD_SCOPED_FILE_URL: an encoded, scoped URL for use inside Snowflake by the caller.
-- It is time-limited -- valid until the persisted query result period ends, currently 24 hours.
SELECT
    RELATIVE_PATH,
    BUILD_SCOPED_FILE_URL('@GENAI_STUDY.PUBLIC.DOCS_STAGE', RELATIVE_PATH) AS scoped_url
FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE)
LIMIT 5;

-- Which to reach for:
--   input to an AI function      -> TO_FILE(...)
--   a Snowflake user needs to look -> BUILD_SCOPED_FILE_URL(...)
--   someone outside Snowflake     -> GET_PRESIGNED_URL(...)   (a bearer credential; keep it short)


### Caching: the cheapest page is the one you do not parse twice

Parsing is billed per page and a document does not change once it is on the stage. The directory
table carries `LAST_MODIFIED`, so "has this file changed since I parsed it?" is a join, not a
guess.

The trade-off is storage and staleness: you are keeping a second copy of every document as text,
and if you ever change parsing mode or Snowflake improves the parser, the cache is the old answer
until you invalidate it deliberately.


In [ ]:
%%sql
-- Optimize: cache parse results to avoid re-parsing the same file
CREATE OR REPLACE TABLE GENAI_STUDY.PUBLIC.PARSE_CACHE (
    filename        VARCHAR(500) PRIMARY KEY,
    last_modified   TIMESTAMP_NTZ,
    parsed_text     VARCHAR(16000),
    parse_mode      VARCHAR(10),
    cached_at       TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Only parse files not yet in cache (or modified since last parse)
INSERT INTO GENAI_STUDY.PUBLIC.PARSE_CACHE (filename, last_modified, parsed_text, parse_mode)
SELECT
    d.RELATIVE_PATH,
    d.LAST_MODIFIED,
    AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', d.RELATIVE_PATH),
        {'mode': 'OCR'}
    ):content::VARCHAR AS parsed_text,
    'OCR'
FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE) d
LEFT JOIN GENAI_STUDY.PUBLIC.PARSE_CACHE c
    ON c.filename = d.RELATIVE_PATH
   AND c.last_modified >= d.LAST_MODIFIED   -- skip if cache is current
WHERE c.filename IS NULL                     -- not in cache
  AND d.RELATIVE_PATH LIKE '%.pdf'
  AND d.SIZE < 104857600;                    -- 100 MB documented limit


In [ ]:
%%sql -r parse_cache_2
SELECT COUNT(*) AS cached_documents FROM GENAI_STUDY.PUBLIC.PARSE_CACHE;


---
## Auditing the privileges

`SNOWFLAKE.ACCOUNT_USAGE` views are the account-wide record of grants, and they lag — up to a couple
of hours. That latency is a trap when you are debugging a grant you made five minutes ago: use
`SHOW GRANTS` for the current state and `ACCOUNT_USAGE` for the wide view.

The definitive test is not reading a grants view at all. It is assuming the role and trying to see
the files.

→ [More on AI function privileges](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


In [ ]:
%%sql -r privilege_check_1
-- Privilege troubleshooting checklist
-- Run as ACCOUNTADMIN to verify all required privileges are in place

-- Check CORTEX_USER is granted
SELECT GRANTEE_NAME, PRIVILEGE, NAME
FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES
WHERE NAME LIKE 'CORTEX%'
  AND DELETED_ON IS NULL;


In [ ]:
%%sql -r privilege_check_2
-- Check stage read access
SELECT GRANTEE_NAME, PRIVILEGE, TABLE_NAME AS stage_name
FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES
WHERE GRANTED_ON = 'STAGE'
  AND TABLE_NAME = 'DOCS_STAGE'
  AND DELETED_ON IS NULL;


In [ ]:
%%sql
-- Test access as the target role
USE ROLE GENAI_ANALYST;


In [ ]:
%%sql -r privilege_check_4
SELECT COUNT(*) FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE);


In [ ]:
%%sql
-- Reset to admin
USE ROLE ACCOUNTADMIN;


---
## When the model is the problem: fine-tuning arctic-extract

`arctic-extract` is the model behind `AI_EXTRACT`. When better questions, LAYOUT mode and
`scale_factor` have all stopped helping — a bespoke claim form, clause identifiers in a house
format, a vendor who redesigned their invoice — you can fine-tune it on your own labelled
documents.

This is a different mechanism from general LLM fine-tuning, which targets `llama3.1-8b` and takes
`prompt`/`completion` text pairs. Reaching for the wrong one is a common exam trap and a common
real-world false start.

### When it is worth it

- Base `AI_EXTRACT` misses domain-specific fields that a human reads instantly
- Accuracy has plateaued despite sharper questions and a higher `scale_factor`
- You can produce at least **20 labelled documents** — and, more to the point, keep producing them
  when the form changes again

### Syntax

```sql
SNOWFLAKE.CORTEX.FINETUNE(
  'CREATE',
  '@<database>.<schema>.<model_name>',     -- the leading @ is required here
  'arctic-extract',                        -- base model
  '<training_dataset>'                     -- table or query with File, Prompt, Response
  [, '<validation_dataset>'                -- fifth POSITIONAL argument
  [, '<options>' ] ]                       -- JSON, e.g. {"max_epochs": 3}
)
```

### Training data — three columns, case-insensitive, any order

| Column | Contents |
|---|---|
| `File` | A string containing the file path to the document, e.g. `@db.schema.stage/invoice_KF-2041.pdf` |
| `Prompt` | A JSON value of key → question pairs |
| `Response` | A JSON object of key → correct answer pairs |

### Limits

| Limit | Value |
|---|---|
| Document formats | PDF, PNG, JPG, JPEG, TIFF, TIF |
| Minimum documents | 20 |
| Maximum unique files | 1,000 |
| Max pages per document | 64 in AWS US West 2 (Oregon) and AWS Europe Central 1 (Frankfurt); 125 in AWS US East 1 (N. Virginia) and Azure East US 2 (Virginia) |
| Max questions | 100 unique for entity extraction, 10 unique for table extraction |
| Questions × total pages across the dataset | ≤ 50,000 |
| `max_epochs` | Integer from 2 through 10 |

### Privileges

`USAGE` or `OWNERSHIP` on the database and schema, `READ` or `OWNERSHIP` on the stages holding the
documents, and `CREATE MODEL` on the target schema.

### Using the result — `model` is a named argument

```sql
SELECT AI_EXTRACT(
    model          => 'GENAI_STUDY.PUBLIC.INVOICE_EXTRACTOR_V2',   -- no @ when calling
    file           => TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_TS-5530.pdf'),
    responseFormat => {'total_amount': 'What is the total amount payable?'}
):response AS fields;
```

`responseFormat` is optional with a fine-tuned model — the model already knows the questions it was
trained on — and supplying one overrides them.

### Managing the job

```sql
SELECT SNOWFLAKE.CORTEX.FINETUNE('SHOW');                 -- takes no arguments
SELECT SNOWFLAKE.CORTEX.FINETUNE('DESCRIBE', '<job_id>'); -- the job id, not the model name
SELECT SNOWFLAKE.CORTEX.FINETUNE('CANCEL',   '<job_id>');
```

**What fine-tuning costs you beyond credits:** a labelled dataset somebody has to produce and keep
current, a model artefact to version and govern, and a new failure mode — a model that quietly
performs worse on document types that were not in the training set.

→ [More on fine-tuning arctic-extract](https://docs.snowflake.com/en/user-guide/snowflake-cortex/arctic-extract-finetuning)


> ### ⚠️ Common misconceptions
>
> **"Fine-tuning for extraction works like fine-tuning a chat model."**
> It does not. Extraction fine-tuning uses the `arctic-extract` base model and a dataset of `File`,
> `Prompt` and `Response` columns. General LLM fine-tuning uses `llama3.1-8b` and `prompt`/
> `completion` text pairs, and the result is called with `AI_COMPLETE`. Bring `prompt`/`completion`
> data to an `arctic-extract` job and the job rejects it.
> → [Fine-tuning arctic-extract](https://docs.snowflake.com/en/user-guide/snowflake-cortex/arctic-extract-finetuning)
>
> **"`FINETUNE('DESCRIBE', 'MY_MODEL')` shows me my model."**
> `DESCRIBE` takes the **job id** returned by `CREATE`, not the model name. Pass a model name and
> you learn nothing about the training run you were trying to monitor.
> → [Fine-tuning arctic-extract](https://docs.snowflake.com/en/user-guide/snowflake-cortex/arctic-extract-finetuning)
>
> **"The model name is written the same way everywhere."**
> `FINETUNE('CREATE', …)` takes the target model with a leading `@` —
> `'@GENAI_STUDY.PUBLIC.INVOICE_EXTRACTOR_V2'`. `AI_EXTRACT` takes it without —
> `model => 'GENAI_STUDY.PUBLIC.INVOICE_EXTRACTOR_V2'`. Getting either wrong fails at the boundary
> between building and using the model.
> → [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)
>
> **"I'll pass epochs and a validation set in the options JSON."**
> The validation dataset is the **fifth positional argument**, not an options key, and the epoch key
> is `max_epochs` with a range of 2 to 10.
> → [Fine-tuning arctic-extract](https://docs.snowflake.com/en/user-guide/snowflake-cortex/arctic-extract-finetuning)


> ### 🤔 Stop and think
>
> - Fine-tuning needs 20 labelled documents to start and a person to keep labelling as forms change.
>   At what accuracy gap does that ongoing human cost beat simply routing low-confidence extractions
>   to a reviewer?
> - `return_error_details => TRUE` makes every failure a queryable row. Nobody is querying it. What
>   would you build so that a 14% failure rate reaches a person — and what is the cost of that alert
>   firing at 3 a.m. for a single corrupt scan?
> - Caching parse results saves real money and creates a second copy of every document's contents
>   inside a normal table, where masking policies and row access policies apply differently than
>   they do to a stage. Is that a governance improvement or a governance problem?


In [ ]:
%%sql -r cost_compare
-- ============================================================
-- AI_PARSE_DOCUMENT is billed PER PAGE, at the same rate for OCR and LAYOUT.
-- Counting output tokens here measures OUTPUT SIZE -- which drives the cost of the
-- NEXT function that reads this text -- not the cost of the parse itself.
-- ============================================================
WITH ocr_result AS (
    SELECT 'OCR' AS mode,
           AI_PARSE_DOCUMENT(TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'financial_statement_mgc_q3_2026.pdf'),
                             {'mode': 'OCR'}):content::VARCHAR AS parsed_text
),
layout_result AS (
    SELECT 'LAYOUT' AS mode,
           AI_PARSE_DOCUMENT(TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'financial_statement_mgc_q3_2026.pdf'),
                             {'mode': 'LAYOUT'}):content::VARCHAR AS parsed_text
),
both AS (SELECT * FROM ocr_result UNION ALL SELECT * FROM layout_result)
SELECT
    mode,
    LENGTH(parsed_text) AS char_count,
    -- AI_COUNT_TOKENS( <function_name>, <model_name>, <input_text> )
    AI_COUNT_TOKENS('AI_COMPLETE', 'llama3.1-8b', parsed_text) AS downstream_input_tokens,
    CASE mode
        WHEN 'OCR'    THEN 'Plain text. Same per-page parse price; smaller downstream prompt.'
        WHEN 'LAYOUT' THEN 'Markdown with tables and headers. Recommended default; larger downstream prompt.'
    END AS guidance
FROM both;

-- The real parsing-cost lever is pages:
-- SELECT AI_PARSE_DOCUMENT(TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE','contract_msa_harbourview.pdf'),
--                          {'mode':'LAYOUT','page_filter':[{'start':0,'end':1}]});


---
## Scenario

**Situation.** A nightly pipeline processes 10,000 PDF invoices with `AI_EXTRACT`. After a vendor
redesigned its invoice template, accuracy on `total_amount` fell from 92% to 61%. Rewording the
question has not helped.

**Question.** What is the Snowflake-native fix, and what does it require?

### Worked solution

Work through the cheap options first. Fine-tuning is the last step, not the first, because it is the
only one with an ongoing human cost.

1. **Find out what kind of wrong it is.** Turn on `scores => TRUE` and look at the confidence on
   `total_amount`. Low confidence points at the model; high confidence with wrong values points at
   the question — the model is confidently answering a different question from the one you meant.
2. **Try LAYOUT** if you were feeding it OCR text. A redesigned invoice usually means a redesigned
   table.
3. **Try `config => {'scale_factor': 2.0}`** if the new template uses smaller type. Remember this
   halves the page ceiling to 62 and raises input tokens proportionally.
4. **Sharpen the question:** *"What is the final total amount payable, including tax, as a number?"*
   rather than *"What is the total?"* on a document that now shows a subtotal, a tax line and a
   balance carried forward.
5. **If it is still low, fine-tune `arctic-extract`.**

**What fine-tuning requires**

- At least **20 labelled documents**, at most 1,000 unique files
- Formats PDF, PNG, JPG, JPEG, TIFF, TIF
- Three columns: `File` (the document path), `Prompt` (JSON of key → question), `Response` (JSON of
  key → correct answer)
- Optionally a validation dataset of the same shape, passed as the **fifth positional argument**
- Page caps of 64 per document in AWS Oregon and Frankfurt, 125 in AWS N. Virginia and Azure
  Virginia; questions × total pages ≤ 50,000
- `CREATE MODEL` on the target schema, plus read access to the stages holding the documents

```sql
-- 1. Training data: File / Prompt / Response
CREATE OR REPLACE TABLE GENAI_STUDY.PUBLIC.EXTRACT_TRAIN AS
SELECT
    '@GENAI_STUDY.PUBLIC.DOCS_STAGE/' || filename AS File,
    OBJECT_CONSTRUCT('total_amount', 'What is the total amount payable?',
                     'vendor_name',  'What company issued this invoice?') AS Prompt,
    OBJECT_CONSTRUCT('total_amount', verified_total,
                     'vendor_name',  verified_vendor)                     AS Response
FROM GENAI_STUDY.PUBLIC.MANUALLY_LABELLED_INVOICES;

-- 2. Fine-tune. The validation set is positional; the option key is max_epochs (2-10).
SELECT SNOWFLAKE.CORTEX.FINETUNE(
    'CREATE',
    '@GENAI_STUDY.PUBLIC.INVOICE_EXTRACTOR_V2',
    'arctic-extract',
    'SELECT File, Prompt, Response FROM GENAI_STUDY.PUBLIC.EXTRACT_TRAIN',
    'SELECT File, Prompt, Response FROM GENAI_STUDY.PUBLIC.EXTRACT_VAL',
    '{"max_epochs": 3}'
) AS job_id;

-- 3. Monitor. DESCRIBE takes the job id returned above, not the model name.
SELECT SNOWFLAKE.CORTEX.FINETUNE('SHOW');
SELECT SNOWFLAKE.CORTEX.FINETUNE('DESCRIBE', '<job_id>');

-- 4. Use it. model is a named argument, and carries no @ at call time.
SELECT AI_EXTRACT(
    model          => 'GENAI_STUDY.PUBLIC.INVOICE_EXTRACTOR_V2',
    file           => TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_TS-5530.pdf'),
    responseFormat => {'total_amount': 'What is the total amount payable?'}
):response AS fields;
```

**What you are signing up for.** Twenty labelled documents is the floor, not the target, and the
vendor will redesign the template again. A cheaper architecture for many teams is to keep the base
model, turn on `scores`, and route anything below a confidence threshold to a human queue — you pay
per exception instead of per template change.


---
## Requirements and privileges — the full checklist

This is the complete set a document pipeline needs, in roughly the order failures surface.

| Layer | Grant or requirement | Failure symptom if missing |
|---|---|---|
| Account | `GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE r;` — or the narrower `GRANT USE AI FUNCTION AI_PARSE_DOCUMENT ON ACCOUNT TO ROLE r;` | AI function calls are refused |
| Cortex | `GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE r;` (or `SNOWFLAKE.AI_FUNCTIONS_USER`) | same |
| Container | `GRANT USAGE ON DATABASE …` and `ON SCHEMA …` | object not found |
| Stage (internal) | `GRANT READ ON STAGE … TO ROLE r;` | access denied, or file not found |
| Stage (external) | `GRANT USAGE ON STAGE … TO ROLE r;` plus a `STORAGE INTEGRATION` | file not found although the object is visibly in the bucket |
| Directory table | `DIRECTORY = (ENABLE = TRUE)`, refreshed with `ALTER STAGE … REFRESH` or `AUTO_REFRESH = TRUE` | `DIRECTORY()` returns nothing; a stream sees no data |
| Compute | `GRANT USAGE ON WAREHOUSE … TO ROLE r;` | the query cannot start |
| Writes | `CREATE TABLE` on the target schema | the pipeline parses and cannot land results |
| Automation | `GRANT EXECUTE TASK ON ACCOUNT TO ROLE r;` (plus `EXECUTE MANAGED TASK` for a serverless task), and the **task owner** holds everything above | the task exists and never produces rows |
| Fine-tuning | `CREATE MODEL` on the target schema, `READ`/`OWNERSHIP` on the document stages | `FINETUNE('CREATE', …)` fails |

Both halves of the AI grant are required: the account privilege **and** one of the two database
roles. `USE AI FUNCTIONS` and `SNOWFLAKE.CORTEX_USER` are granted to `PUBLIC` by default, which is
why this often appears to need no setup until you meet an account where the `PUBLIC` grant was
revoked. `SNOWFLAKE.AI_FUNCTIONS_USER` is the narrower role — scalar functions only, no `AI_AGG` or
`AI_SUMMARIZE_AGG` — and is not granted to `PUBLIC`.

### Stages that will never work

AI functions cannot build a FILE object over a **user stage** (`@~`), a **table stage** (`@%tbl`),
an internal stage with `TYPE = 'SNOWFLAKE_FULL'` encryption, an external stage using a customer-side
encryption mode such as `AWS_CSE` or `AZURE_CSE`, or a stage with a double-quoted name.
`AI_PARSE_DOCUMENT` itself supports documents on stages using client-side or server-side encryption,
including in accounts using PrivateLink — the restrictions above are on the FILE object, and they
are what you are hitting when a valid path reports an access error.

### Owner's rights

A stored procedure or a task runs as its **owner**, not the caller. If a pipeline works
interactively and silently does nothing on a schedule, check the owner role's grants before
anything else.

→ [AI function privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


In [ ]:
%%sql -r ctx_check
-- 1. Does the current role have what it needs? Start here.
SELECT CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS warehouse, CURRENT_DATABASE() AS db;

In [ ]:
%%sql -r role_grants_check
-- 2. Cortex database roles reaching this role
SHOW GRANTS TO ROLE IDENTIFIER(CURRENT_ROLE());

In [ ]:
%%sql -r stage_grants_check
-- 3. Stage privileges actually granted (ACCOUNT_USAGE has up to ~2h latency)
SELECT GRANTEE_NAME, PRIVILEGE, TABLE_NAME AS stage_name, GRANTED_ON
FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES
WHERE GRANTED_ON = 'STAGE'
  AND DELETED_ON IS NULL
ORDER BY GRANTEE_NAME;

In [ ]:
%%sql -r directory_check
-- 4. Can the role actually see files? This is the real test — it exercises the whole chain.
SELECT COUNT(*) AS visible_files FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE);

---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** How do you get an error message instead of a `NULL` from `AI_PARSE_DOCUMENT`?

<details><summary>Show answer</summary>

Pass `TRUE` as the third, positional `return_error_details` argument. The result becomes
`{"value": …, "error": …, "metadata": …}`. It is not a key inside the options object, and there is
no `TRY_AI_PARSE_DOCUMENT` — that function does not exist.

→ [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)

</details>

**2.** Which three columns does an `arctic-extract` training dataset need, and how many documents
at minimum?

<details><summary>Show answer</summary>

`File`, `Prompt` and `Response` — case-insensitive and in any order — with at least 20 documents and
no more than 1,000 unique files. `File` is a string path to the document, `Prompt` is JSON of key →
question, `Response` is JSON of key → correct answer. `prompt`/`completion` is the schema for
general LLM fine-tuning on `llama3.1-8b`, which is a different feature.

→ [Fine-tuning arctic-extract](https://docs.snowflake.com/en/user-guide/snowflake-cortex/arctic-extract-finetuning)

</details>

**3.** What is the range of `max_epochs`, and where is the validation dataset passed?

<details><summary>Show answer</summary>

`max_epochs` is an integer from 2 through 10, supplied inside the options JSON. The validation
dataset is not an option at all — it is the fifth positional argument to `FINETUNE`, between the
training dataset and the options string.

→ [Fine-tuning arctic-extract](https://docs.snowflake.com/en/user-guide/snowflake-cortex/arctic-extract-finetuning)

</details>

**4.** A nightly batch inserts 4,000 rows and 500 have `NULL` text. The query never errored. What
happened, and how would you have known sooner?

<details><summary>Show answer</summary>

`AI_PARSE_DOCUMENT` returns `NULL` on failure by default, so 500 documents failed silently and
inserted as empty rows. With `return_error_details => TRUE` each failure would have carried an
`error` string you could group and count. Nothing about the statement's success tells you anything
about the documents.

→ [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)

</details>

**5.** `GET_PRESIGNED_URL` returns a URL for `invoice_KF-2041.pdf`, but opening it gives an error.
Does that prove the privileges are fine?

<details><summary>Show answer</summary>

No. The function generates a URL even when the file does not exist on the stage, so a successful
call proves only that you could call the function. Check existence with the directory table. It also
requires server-side encryption on the stage, so a failure to fetch can be a stage configuration
problem rather than a missing file or a missing grant.

→ [GET_PRESIGNED_URL](https://docs.snowflake.com/en/sql-reference/functions/get_presigned_url)

</details>

**6.** A team runs `FINETUNE('CREATE', 'GENAI_STUDY.PUBLIC.MY_MODEL', 'arctic-extract', …)` and it
fails. What is wrong?

<details><summary>Show answer</summary>

The target model name needs a leading `@`: `'@GENAI_STUDY.PUBLIC.MY_MODEL'`. The asymmetry is worth
memorising, because `AI_EXTRACT` takes the same model *without* the `@` —
`model => 'GENAI_STUDY.PUBLIC.MY_MODEL'`.

→ [Fine-tuning arctic-extract](https://docs.snowflake.com/en/user-guide/snowflake-cortex/arctic-extract-finetuning)

</details>

**7.** Extraction accuracy on a scanned form is poor. Someone sets `scale_factor` to 4.0 across a
pipeline of 100-page documents. What breaks?

<details><summary>Show answer</summary>

At `scale_factor` 4.0 the page ceiling falls to 31, so every 100-page document is rejected outright.
Input token consumption also rises in proportion to the scale. The range is 1.0 through 4.0 with a
default of 1.0; raise it for the documents that need it, not as a global setting.

→ [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)

</details>

**8.** A 250 MB, 300-page scanned contract must be processed. Which limit bites, and does
`page_filter` rescue it?

<details><summary>Show answer</summary>

The 100 MB file size limit. The 2,000-page limit is nowhere near. `page_filter` does not help,
because the size limit applies to the file being read, not to the pages processed — the file has to
be split before Snowflake will accept it at all.

→ [Parsing documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)

</details>

**9.** Someone proposes switching a 2-million-page annual workload from LAYOUT to OCR "to halve the
parsing bill". Assess it.

<details><summary>Show answer</summary>

It will not halve anything: both modes are billed per page at the same rate. The change would
shrink the parsed text, which lowers the input tokens of any downstream AI function — a real saving,
but a different one, and it costs table structure that LAYOUT preserves. The levers that would
actually move a 2-million-page bill are `page_filter`, caching so unchanged files are never
re-parsed, and not parsing at all when `AI_EXTRACT` on the file would do.

→ [Parsing documents: cost considerations](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)

</details>

**10.** You granted `READ ON STAGE` five minutes ago and `ACCOUNT_USAGE.GRANTS_TO_ROLES` does not
show it. Is the grant missing?

<details><summary>Show answer</summary>

Probably not. `ACCOUNT_USAGE` views have latency — commonly up to a couple of hours. Use
`SHOW GRANTS TO ROLE <role>` for the current state. The conclusive test is neither view: assume the
role and run `SELECT COUNT(*) FROM DIRECTORY(@stage)`, which exercises the whole chain the AI
function will walk.

→ [AI function privileges](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**11.** Accuracy on one vendor's invoices is 61%. Fine-tuning would likely fix it. Argue for doing
something else first.

<details><summary>Show answer</summary>

Fine-tuning carries an ongoing human cost — at least 20 labelled documents now, and more every time
the template changes — plus a model artefact to version and govern, and the risk of regressing on
document types absent from the training set. Turning on `scores => TRUE` and routing low-confidence
extractions to a review queue costs per exception instead of per template change, and gives you the
labelled data as a by-product if you later decide to fine-tune anyway.

→ [Fine-tuning arctic-extract](https://docs.snowflake.com/en/user-guide/snowflake-cortex/arctic-extract-finetuning)

</details>

**12.** A document pipeline works in your worksheet and produces nothing when its task runs
overnight. Walk the diagnosis.

<details><summary>Show answer</summary>

Start with state, not code. Is the task resumed — tasks are created suspended, and resuming a root
does not resume its children. Then the run history: `SKIPPED` means the `WHEN` gate was false, which
usually means the directory table was never refreshed so the stream saw nothing; `FAILED` gives you
the message; no rows at all means the task never fired. If it ran and failed, look at the **task
owner's** privileges, because a task executes as its owner and needs `EXECUTE TASK`, warehouse
`USAGE`, the AI grants and stage access in its own right. This is the same owner's-rights rule that
governs the stored procedure in 4.3.

→ [Task graphs](https://docs.snowflake.com/en/user-guide/tasks-graphs)

</details>
